In [1]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)

**Input**

In [2]:
batch_size = 1
seq_len = 4
d_model = 8

x = torch.tensor(
    [
        [
            [1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0],
            [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0],
            [1.0, 1.0, 1.0, 1.0, 0.5, 0.5, 0.5, 0.5],
            [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2]
        ]
    ],
    dtype=torch.float32
)

print("Input shape:", x.shape)
print(x)

Input shape: torch.Size([1, 4, 8])
tensor([[[1.0000, 0.0000, 1.0000, 0.0000, 1.0000, 0.0000, 1.0000, 0.0000],
         [0.0000, 1.0000, 0.0000, 1.0000, 0.0000, 1.0000, 0.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]]])


In [3]:
def print_tensor(name, tensor):
    print(f"\n{name} shape: {tensor.shape}")
    print(tensor)

**Padding mask function**
- 1 = Valid token
- 0 = pad token

In [4]:
def make_padding_mask(valid_tokens):
    mask = valid_tokens.unsqueeze(1).unsqueeze(2)
    return mask

**Causal mask function**

In [5]:
def make_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
    mask = mask.unsqueeze(0).unsqueeze(0)
    return mask

**Combined mask helper**

In [6]:
def combine_masks(padding_mask=None, causal_mask=None):
    if padding_mask is None:
        return causal_mask
    if causal_mask is None:
        return padding_mask
    return padding_mask & causal_mask

**Mask-aware scaled dot-product attention**

In [7]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    print("\n===== scaled_dot_product_attention =====")
    print("Q shape:", Q.shape)
    print("K shape:", K.shape)
    print("V shape:", V.shape)
    
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    print_tensor("Raw attention scores", scores)

    if mask is not None:
        print_tensor("Mask", mask)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        print_tensor("Masked Scores", scores)
    
    attention_weights = torch.softmax(scores, dim=-1)
    print_tensor("Attention weights", attention_weights)

    output = torch.matmul(attention_weights, V)
    print_tensor("Attention output", output)

    return output, attention_weights

**Q, K, V projection layers**

In [8]:
num_heads = 2
head_dim = d_model // num_heads

W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model, bias=False)

**Split and combine heads**

In [9]:
def split_heads(x, num_heads):
    batch_size, seq_len, d_model = x.shape
    head_dim = d_model // num_heads
    x = x.view(batch_size, seq_len, num_heads, head_dim)
    x = x.transpose(1, 2)
    return x

def combine_heads(x):
    batch_size, num_heads, seq_len, head_dim = x.shape
    x = x.transpose(1, 2).contiguous()
    x = x.view(batch_size, seq_len, num_heads * head_dim)
    return x

#### No Mask Case

In [10]:
Q = W_q(x)
K = W_k(x)
V = W_v(x)

Q_heads = split_heads(Q, num_heads)
K_heads = split_heads(K, num_heads)
V_heads = split_heads(V, num_heads)

print_tensor("Q_heads", Q_heads)
print_tensor("K_heads", K_heads)
print_tensor("V_heads", V_heads)

output_heads, attention_weights = scaled_dot_product_attention(Q_heads, K_heads, V_heads, mask=None)

combined_output = combine_heads(output_heads)
final_output = W_o(combined_output)

print_tensor("Combined output", combined_output)
print_tensor("Final output", final_output)


Q_heads shape: torch.Size([1, 2, 4, 4])
tensor([[[[-0.0621,  1.0507, -0.1990, -0.0263],
          [ 0.8972, -0.1952,  0.3355, -1.0251],
          [ 0.8204,  0.6406,  0.1932, -0.9029],
          [ 0.1670,  0.1711,  0.0273, -0.2103]],

         [[ 0.2519,  0.6523, -0.3398,  0.5198],
          [ 0.4515,  0.1285,  0.0845, -0.6204],
          [ 0.5694,  0.6222, -0.3889,  0.0153],
          [ 0.1407,  0.1561, -0.0511, -0.0201]]]],
       grad_fn=<TransposeBackward0>)

K_heads shape: torch.Size([1, 2, 4, 4])
tensor([[[[ 0.5087, -0.1217,  0.4218, -0.1493],
          [-0.3763, -0.1314, -0.7425,  0.0692],
          [ 0.1254, -0.4160, -0.3498, -0.0831],
          [ 0.0265, -0.0506, -0.0641, -0.0160]],

         [[-0.0507, -0.2477, -0.2203, -0.3872],
          [-0.6297, -0.5945, -0.9702, -0.8565],
          [-0.4717, -0.7850, -0.8806, -0.8192],
          [-0.1361, -0.1684, -0.2381, -0.2487]]]],
       grad_fn=<TransposeBackward0>)

V_heads shape: torch.Size([1, 2, 4, 4])
tensor([[[[ 2.0397e-01,  

### Padding mask case

- let valid tokens = [1, 1, 1, 0]

In [11]:
valid_tokens = torch.tensor([[1, 1, 1, 0]], dtype=torch.bool)

padding_mask = make_padding_mask(valid_tokens)
print_tensor("Padding mask", padding_mask)

output_heads_pad, attention_weights_pad = scaled_dot_product_attention(
    Q_heads, K_heads, V_heads, mask=padding_mask
)

combined_output_pad = combine_heads(output_heads_pad)
final_output_pad = W_o(combined_output_pad)

print_tensor("Final output with padding mask", final_output_pad)


Padding mask shape: torch.Size([1, 1, 1, 4])
tensor([[[[ True,  True,  True, False]]]])

===== scaled_dot_product_attention =====
Q shape: torch.Size([1, 2, 4, 4])
K shape: torch.Size([1, 2, 4, 4])
V shape: torch.Size([1, 2, 4, 4])

Raw attention scores shape: torch.Size([1, 2, 4, 4])
tensor([[[[-0.1197,  0.0156, -0.1865, -0.0208],
          [ 0.3874, -0.3160,  0.0808,  0.0143],
          [ 0.2778, -0.2994, -0.0781, -0.0043],
          [ 0.0535, -0.0601, -0.0212, -0.0013]],

         [[-0.1504, -0.3310, -0.3787, -0.0963],
          [ 0.0835,  0.0444,  0.0600,  0.0256],
          [-0.0516, -0.1821, -0.2135, -0.0467],
          [-0.0134, -0.0573, -0.0637, -0.0141]]]], grad_fn=<DivBackward0>)

Mask shape: torch.Size([1, 1, 1, 4])
tensor([[[[ True,  True,  True, False]]]])

Masked Scores shape: torch.Size([1, 2, 4, 4])
tensor([[[[-0.1197,  0.0156, -0.1865,    -inf],
          [ 0.3874, -0.3160,  0.0808,    -inf],
          [ 0.2778, -0.2994, -0.0781,    -inf],
          [ 0.0535, -0.0601,